In [ ]:
import pandas as pd
from dash import Dash, dcc, html, Input, Output, dash_table
import plotly.express as px
import numpy as np



# --- Charger l'Excel ---
file_path = r"C:\Users\ymalier\OneDrive - ALEDIA\Documents Pro\python\VLCDesignTool\src\assets\ressources\VLC_Samples_Data_Model.xlsx"

df_LIV = pd.read_excel(file_path, sheet_name="LIV")
df_LIV = df_LIV.sort_values("Voltage (V)")
df_valid = df_LIV[df_LIV["EQE (%)"] > 0]

Jmin = df_valid["J (A/cm²)"].min()
Jmax = df_valid["J (A/cm²)"].max()

df_f3db = pd.read_excel(file_path, sheet_name="f-3dB")

# --- Liste des échantillons ---
samples = df_LIV["Sample"].unique()
tests = df_f3db["Test"].unique()

def interp(df, J, xcol, ycol):
        try:
            # dataframe vide → erreur → on renvoie 0
            if df.empty:
                return 0.0
            
            # interpolation brute
            val = float(np.interp(J, df[xcol], df[ycol]))

            # si NaN → renvoyer 0
            if np.isnan(val):
                return 0.0

            return val

        except Exception:
            return 0.0



app = Dash(__name__)

app.layout = html.Div([
    html.H1("VLC system Insight – From manual Excel Data"),

    dcc.Dropdown(id="sample-select",
                 options=[{"label": s, "value": s} for s in samples],
                 value=samples[0],
                 clearable=False),

    dcc.Dropdown(id="test-select",
                 options=[{"label": t, "value": t} for t in tests],
                 value=tests[0],
                 clearable=False),

    html.Div([html.Label("J0 level - bit 0"),dcc.Input(id="j1-input", type="number", placeholder="Entrer J0 level for bit 0", step=0.001, value=0.2),]),
    html.Div([html.Label("J1 level - bit 1"),dcc.Input(id="j2-input", type="number", placeholder="Entrer J1 level for bit 1", step=0.001, value=100),]),
    # --- 4 graphes sur une ligne ---
    html.Div([
        dcc.Graph(id="eqe-graph", style={"width": "25%", "display": "inline-block"}),
        dcc.Graph(id="wopt-graph", style={"width": "25%", "display": "inline-block"}),
        dcc.Graph(id="v-graph", style={"width": "25%", "display": "inline-block"}),
        dcc.Graph(id="f3db-graph", style={"width": "25%", "display": "inline-block"}),
    ], style={"display": "flex", "flexDirection": "row"}),


    html.Div([

        html.Div([
        html.H3("µLEDs driver"),
        html.Div("Consumption driver model (pJ/bit):", style={"fontWeight": "bold"}),
        html.Div([
            "E_driver = P_static / Rb + C_cmos_grid × V_logic² + C_DS × (V_1-V_0)²",
            html.Br(),
            "• Rb : Bit rate in bit per s",
            html.Br(),
            "• Logic Voltage : Source-grid swing voltage",
            html.Br(),
            "• C_DS : Drain-source CMOS capacitance"
        ], style={"fontSize": "12px", "marginBottom": "10px"}),
        html.Div([html.Label("Static consumption (mW)"),dcc.Input(id="static_driver-input", type="number",placeholder="Static consumption (mW)", step=0.0001, value=0.02),]),
        html.Div([html.Label("CMOS Grid capacitance (fF)"),dcc.Input(id="capa-cmos-input", type="number",placeholder="Load capacitance (fF)", step=1, value=50),]),
        html.Div([html.Label("V_logic (V)"),dcc.Input(id="logicV-input", type="number", placeholder="Logic Voltage (V)", step=0.01, value=1.2),]),
        html.Div([html.Label("Capacitance drain-source (fF)"),dcc.Input(id="Cds-input", type="number", placeholder="Drain source capacitance (V)", step=0.01, value=50),]),
        html.Div([html.Label("Time bit window (ns)"),dcc.Input(id="twindow-input", type="number", placeholder="Twindow (ns)", step=0.01, value=1),]),
        html.Div([html.Label("Bit rate (Gbit/s)"),dcc.Input(id="bitrate-input", type="number", placeholder="Rb (bit/s)", step=0.001, value=0.125),]),
        html.Div([html.Label("Rise time (ns)"),dcc.Input(id="rtime-input", type="number", placeholder="Rise time (ns)", step=0.001, value=0.5),]),
        html.Div([html.Label("Fall time (ns)"),dcc.Input(id="ftime-input", type="number", placeholder="Fall time (ns, optionnel)", step=0.001, value=0.5),]),],
        style={"border": "1px solid #ccc", "padding": "10px", "margin": "10px", "flex": "1"}),

        html.Div([
        html.H3("µLEDs array and coupling"),        
        html.Div("Consumption LED model (pJ/bit):", style={"fontWeight": "bold"}),
        html.Div([
            "E_led = I × V × twindow",
            html.Br(),
            "• twindow : Bit window duration",
            html.Br(),
            "• I,V : cuurent, voltage drive"
        ], style={"fontSize": "12px", "marginBottom": "10px"}),
        html.Div([html.Label("Number of wires per LED"),dcc.Input(id="n-wires-input", type="number",placeholder="Nombre de fils", min=1, step=1, value=4),]),        
        html.Div([html.Label("λ (nm)"),dcc.Input(id="lambda-input", type="number", placeholder="λ photon (nm)", step=1, value=450),]),
        html.Div([html.Label("Coupling LED - Photodetector (% of SL)"),dcc.Input(id="coupling-input", type="number",placeholder="Couplage LED→PD (%)", step=0.01, value=4),]),
        html.Div([html.Label("Xtalk on Photodetector (ppm)"),dcc.Input(id="xtalk-input", type="number", placeholder="Xtalk voisins (ppm)", step=0.000001, value=0.1),]),
        html.Div([html.Label("Pitch LED x (µm)"),dcc.Input(id="xpitch-input", type="number", placeholder="x pitch (µm)", step=1, value=22),]),
        html.Div([html.Label("Pitch LED y (µm)"),dcc.Input(id="ypitch-input", type="number", placeholder="y pitch (µm)", step=1, value=22),]),],
        style={"border": "1px solid #ccc", "padding": "10px", "margin": "10px", "flex": "1"}),

        html.Div([
        html.H3("Photodetector"),
        html.Div("Photodetector type:", style={"fontWeight": "bold"}),
        dcc.Dropdown(
            id="rx-type-select",
            options=[
                {"label": "SPAD", "value": "SPAD"},
                {"label": "APD", "value": "APD"}
            ],
            value="SPAD",
            clearable=False
        ),
        html.Div([html.Label("Ratio detector & illumination spot area"),dcc.Input(id="ratiodetectorspot-input", type="number", placeholder="Rdet-spot", step=0.0001, value=0.17),]),
        html.Div([html.Label("Number of detectors"),dcc.Input(id="ndetector-input", type="number", placeholder="Ndet", step=1, value=2),]),
        # --- Bloc SPAD ---
        html.Div(id="spad-block", children=[
            html.Div("Consumption spad model (pJ/bit):", style={"fontWeight": "bold"}),
            html.Div("E_photodetector = N × Qavalanche × Vbias"),
            html.Div([html.Label("Qavalanche (fC)"), dcc.Input(id="qavalanche-input", type="number", value=20),]),
            html.Div([html.Label("Vbias (V)"),dcc.Input(id="vbias-input", type="number", value=16),]),
            html.Div("BER spad model:", style={"fontWeight": "bold"}),
            html.Div([
                html.Ul([
                    html.Li("BER = 0.5 × (Pmiss + Pfalse)"),
                    html.Li("Pmiss = 1 − Pdetⁿ (n = number of detectors)"),
                    html.Li("Pdet = 1 − exp(− PDE × Flux × Twindow / Ephoton)"),
                    html.Li("Pfalse = (1 − exp(− DCR × Twindow − PDE × Flux_XT × Twindow / Ephoton))ⁿ"),
                ], style={"fontSize": "13px"})
            ]),
            html.Div([html.Label("PDE (%)"),dcc.Input(id="pde-input", type="number", placeholder="PDE (%)", step=0.5, value=20),]),
            html.Div([html.Label("DCR (Hz ie count per s)"),dcc.Input(id="dcr-input", type="number", placeholder="DCR (Hz)", step=10, value=1000),]),
        ]),

        # --- Bloc APD ---
        html.Div(id="apd-block", children=[
            html.Div("BER APD model:", style={"fontWeight": "bold"}),
            html.Ul([
                html.Li("BER = 0.5 × erfc(Q-factor/sqrt(2))"),
                html.Li("Q-factor = ..."),
            ]),
        ]),
        
        ], style={"border": "1px solid #ccc", "padding": "10px", "margin": "10px", "flex": "1"}),
        html.Div([
        html.H3("Readout amplifier"),
        html.Div("Consumption model (pJ/bit):", style={"fontWeight": "bold"}),
        html.Div([
            "E_photodetector = P_ampli / Rb",
            html.Br(),
            "• Rb : Bit rate in bit per s",
        ], style={"fontSize": "12px", "marginBottom": "10px"}),
        html.Div([html.Label("Readout amplifier power (mW)"),dcc.Input(id="ampli_power", type="number", placeholder="Ampli power (mW)", step=0.00001, value=5e-3),]),
        ], style={"border": "1px solid #ccc", "padding": "10px", "margin": "10px", "flex": "1"}),
    ], style={"display": "flex", "flex-direction": "row"}),
    


    html.H2("Summary table LIV - f-3dB - pJ/bit - BER"),
    dash_table.DataTable(
        id="summary-table",
        columns=[
            {"name": "Paramètre", "id": "param"},
            {"name": "J1", "id": "j1"},
            {"name": "J2", "id": "j2"},
        ],
        data=[],
        style_table={"marginTop": "20px", "width": "50%"},
        style_cell={"textAlign": "center"},
    ),



])

h = 6.626e-34
c = 3e8

def ephoton_from_lambda_nm(lambda_nm):
    lam = (lambda_nm or 450) * 1e-9
    return h * c / lam


def BER_OOK_Ndet_AND(P0, P1, PDE, DCR, Twindow, Eph, Ndetector, Rdetspot):
    Nph0 = P0 * Rdetspot * Twindow / Eph
    Nph1 = P1 * Rdetspot * Twindow / Eph

    Pdet1 = 1 - np.exp(-PDE * Nph1 )
    Pnoise = (1 - np.exp(-(DCR * Twindow + PDE * Nph0 )))

    Pmiss = 1 - Pdet1**Ndetector
    Pfalse = Pnoise**Ndetector

    return 0.5 * (Pmiss + Pfalse), Pmiss, Pfalse, Nph0, Nph1


# --- Callback principal ---
@app.callback(
    Output("eqe-graph", "figure"),
    Output("wopt-graph", "figure"),
    Output("v-graph", "figure"),
    Output("f3db-graph", "figure"),
    Output("summary-table", "data"),
    Input("sample-select", "value"),
    Input("test-select", "value"),
    Input("j1-input", "value"),
    Input("j2-input", "value"),
    Input("static_driver-input","value"),
    Input("capa-cmos-input","value"),
    Input("logicV-input", "value"), 
    Input("Cds-input", "value"), 
    Input("twindow-input", "value"), 
    Input("bitrate-input", "value"), 
    Input("rtime-input", "value"), 
    Input("ftime-input", "value"),
    Input("n-wires-input", "value"),
    Input("coupling-input", "value"),
    Input("xtalk-input", "value"),    
    Input("xpitch-input", "value"),    
    Input("ypitch-input", "value"),
    Input("rx-type-select", "value"),
    Input("ratiodetectorspot-input", "value"),
    Input("ndetector-input", "value"),
    Input("qavalanche-input", "value"),
    Input("vbias-input", "value"),
    Input("lambda-input", "value"),
    Input("pde-input", "value"),
    Input("dcr-input", "value"),       
    Input("ampli_power", "value"),
)

def update_graphs(sample, test, j1, j2, static_driver,capaCMOS,logicV, Cds, twindow_ns, bitrate_bit_s, rtime, ftime, n_wires, coupling, xtalk, xpitch, ypitch, rx_type, ratiodetectorspot, ndetector, qavalanche, vbias, lambda_nm, pde_pct, dcr, ampli_power):

    df1 = df_LIV[df_LIV["Sample"] == sample].copy()
    df2 = df_f3db[(df_f3db["Sample"] == sample) & (df_f3db["Test"] == test)]

    df1["L per wire (µW)"]=df1["L (W)"]*1e6/float(samples[0].split("_")[3].split("-")[0])
 


    # --- Graphes ---
    fig_eqe = px.line(df1, x="J (A/cm²)", y="EQE (%)", markers=True, title=f"EQE vs J ({sample})", color_discrete_sequence=["black"])
    fig_eqe.update_xaxes(type="log")
    fig_eqe.update_xaxes(type="log", range=[-1, None])

    fig_wopt = px.line(df1, x="J (A/cm²)", y="L per wire (µW)", markers=True, title=f"Wopt vs J ({sample})", color_discrete_sequence=["black"])
    fig_wopt.update_xaxes(type="log")
    fig_wopt.update_yaxes(type="log")    
    fig_wopt.update_xaxes(type="log", range=[-1, None])

    fig_v = px.line(df1, x="Voltage (V)", y="J (A/cm²)", markers=True, title=f"V vs J ({sample})", color_discrete_sequence=["black"])
    fig_v.update_yaxes(type="log")

    fig_f3db = px.line(df2, x="J (A/cm²)", y="f-3dB (MHz)", markers=True, title=f"f-3dB vs J ({sample})", color_discrete_sequence=["black"])
    fig_f3db.update_xaxes(type="log")
    fig_f3db.update_yaxes(type="log")

    # --- Lignes verticales ---
    for fig in [fig_eqe, fig_wopt, fig_f3db]:
        if j1 is not None:
            fig.add_vline(x=j1, line_width=2, line_dash="dash", line_color="green")
        if j2 is not None:
            fig.add_vline(x=j2, line_width=2, line_dash="dash", line_color="blue")

    # --- Lignes horizontales pour IV ---
    if j1 is not None:
        fig_v.add_hline(y=j1, line_width=2, line_dash="dash", line_color="green")
    if j2 is not None:
        fig_v.add_hline(y=j2, line_width=2, line_dash="dash", line_color="blue")

    # --- Interpolation ---
    summary = []
    if j1 is not None and j2 is not None:

        V1 = interp(df1, j1, "J (A/cm²)", "Voltage (V)")
        l1 = interp(df1, j1, "J (A/cm²)", "L per wire (µW)")
        L1 = interp(df1, j1, "J (A/cm²)", "L (W)")
        EQE1 = interp(df1, j1, "J (A/cm²)", "EQE (%)")
        f3db1 = interp(df2, j1, "J (A/cm²)", "f-3dB (MHz)")

        V2 = interp(df1, j2, "J (A/cm²)", "Voltage (V)")
        l2 = interp(df1, j2, "J (A/cm²)", "L per wire (µW)")        
        L2 = interp(df1, j2, "J (A/cm²)", "L (W)")
        EQE2 = interp(df1, j2, "J (A/cm²)", "EQE (%)")
        f3db2 = interp(df2, j2, "J (A/cm²)", "f-3dB (MHz)")

        # 
        R_time = (rtime*1e-9 or 1e-20)
        F_time = (ftime*1e-9 or 1e-20)
        p_static_drive=static_driver*1e-3
        Cdrive=capaCMOS*1e-15
        Vdrive=logicV
        k_cpl = (coupling or 0.0) / 100.0
        k_xt  = (xtalk*coupling or 0.0) *1e-6 / 100.0
        N     = int(n_wires or 1)
        C_DS = Cds * 1e-15

        PDE = (pde_pct or 0.0) / 100.0
        DCR = dcr or 100.0
        Ndetector = (ndetector or 0)
        R_detector_spot = (ratiodetectorspot or 1)
        Qavalanche = (qavalanche or 50.0) * 1e-15
        Vbias = vbias or 3.0
        lambda_nm = lambda_nm or 450.0
        Eph = ephoton_from_lambda_nm(lambda_nm)
        P_ampli = (ampli_power*1e-3  or 0.0)
        Rb = (bitrate_bit_s * 1e9  or 0.0)
        # Twindow : si non donné, on prend 1 / Rb
        
        if twindow_ns is not None:
            Twindow = twindow_ns * 1e-9
        else:
            Twindow = 1.0 / Rb

        # --- Puissance optique par fil (bit 0 et 1) ---
        # L1, L2 sont déjà "per wire (µW)"
        P1_wire = l2 * N      # µW pour N fils à 1
        P0_wire = l1 * N      # µW pour N fils à 0 (si tu veux un 0 non nul)

        # Surface active totale (cm²)
        S = N * 25* 1e-8

        # --- Puissance instantannée moyenne reçue utile (bit 1) ---
        P_sig_1_uW = P1_wire * k_cpl * (Twindow-R_time*(1-np.exp(-Twindow/R_time)))/Twindow
        P_sig_0_uW = P0_wire * k_cpl + P1_wire * k_cpl * F_time*(np.exp(-((1/Rb)-Twindow)/F_time)-np.exp(-1/(Rb*F_time))) / Twindow
        print(P1_wire * k_cpl * F_time*(np.exp(-((1/Rb)-Twindow)/F_time)-np.exp(-1/(Rb*F_time))) / Twindow)
        # --- Xtalk : tous les voisins à 1 ---
        P_xtalk_uW = P1_wire * k_xt

        # Puissance au PD (µW)
        P_PD_1_uW = P_sig_1_uW + P_xtalk_uW
        P_PD_0_uW = P_sig_0_uW + P_xtalk_uW   # si le fil utile est à 0 mais voisins à 1

        # Conversion en W
        P_PD_1 = P_PD_1_uW * 1e-6
        P_PD_0 = P_PD_0_uW * 1e-6

        # --- Courant LED ---
        I2 = j2 * S     # A
        P_elec_2 = V2 * I2   # W
        I1 = j1 * S     # A
        P_elec_1 = V1 * I1   # W

        # --- Énergie LED ---
        E_led_pJ_2 = P_elec_2 * Twindow * 1e12
        E_led_pJ_1 = P_elec_1 * Twindow * 1e12

        # --- Énergie Driver ---
        E_driver=(p_static_drive/Rb+Cdrive*Vdrive**2+C_DS*(V2-V1)**2)*1e12
        E_ampli_pJ = P_ampli / Rb * 1e12

        # --- Énergie Detecteur + Ampli ---
        if rx_type == "APD":
            E_detector = Ndetector * Qavalanche * Vbias * 1e12
        elif rx_type == "SPAD":
            E_detector = Ndetector * Qavalanche * Vbias * 1e12
        else:
            E_detector = 0
            E_ampli_pJ = 0.0

        E_total_pJ_1 = E_driver + E_led_pJ_1 + E_detector + E_ampli_pJ
        E_total_pJ_2 = E_driver + E_led_pJ_2 + E_detector + E_ampli_pJ

        # --- BER selon le type de récepteur ---
        if rx_type == "APD":
            ber=1e-9
        elif rx_type == "SPAD":
            ber,pmiss,pfalse, Nph0, Nph1 = BER_OOK_Ndet_AND(P_PD_0, P_PD_1, PDE, DCR, Twindow, Eph, Ndetector, R_detector_spot)
        else:
            # Placeholder APD / APD+Ampli
            ber = 1

        Tbit_s_mm_2 = Rb*1e-12/(xpitch*ypitch*1e-6)

        summary = [
            {"param": "J (A/cm²)", "j1": format(j1, ".0f"), "j2": format(j2, ".0f")},
            {"param": "Voltage (V)", "j1": format(V1,".2f"), "j2": format(V2,".2f")},
            {"param": "L per wire (µW)", "j1": format(l1,".2f"), "j2": format(l2,".2f")},            
            {"param": "L Nwires (µW)", "j1": format(l1*N,".2f"), "j2": format(l2*N,".2f")},                        
            {"param": "L on each spad (µW)", "j1": format(l1*N*k_cpl,".2f"), "j2": format(l2*N*k_cpl,".2f")},                        
            {"param": "Nph on each spad", "j1": format(Nph0,".0f"), "j2": format(Nph1,".0f")},
            {"param": "EQE (%)", "j1": format(EQE1,".2f"), "j2": format(EQE2,".2f")},
            {"param": "f-3dB (MHz)", "j1": format(f3db1, ".1f"), "j2": format(f3db2,".1f")},            
            {"param": "Energy (pJ/bit)", "j1": f"{E_total_pJ_1:.3e}", "j2": f"{E_total_pJ_2:.3e}"},
            {"param": "Contribution (% of tot pJ/bit)", "j1": f"Driver: {100*E_driver/E_total_pJ_1:.1f}% - LED: {100*E_led_pJ_1/E_total_pJ_1:.1f}% – Detector: {100*E_detector/E_total_pJ_1:.1f}% – Ampli: {100*E_ampli_pJ/E_total_pJ_1:.1f}%", "j2": f"Driver: {100*E_driver/E_total_pJ_2:.1f}% - LED: {100*E_led_pJ_2/E_total_pJ_2:.1f}% – Detector: {100*E_detector/E_total_pJ_2:.1f}% – Ampli: {100*E_ampli_pJ/E_total_pJ_2:.1f}%"},
            {"param": "BER", "j2": format(ber,".3e")},
            {"param": "Contribution (% of tot ber)", "j2": f"1 Miss: {100*0.5*pmiss/ber:.1f}% - 1 False: {100*0.5*pfalse/ber:.1f}%"},
            {"param": "TBit/s/mm²",  "j2": format(Tbit_s_mm_2,".3f")},
        ]



    return fig_eqe, fig_wopt, fig_v, fig_f3db, summary

@app.callback(
    Output("spad-block", "style"),
    Output("apd-block", "style"),
    Input("rx-type-select", "value")
)
def toggle_detector_blocks(detector_type):
    if detector_type == "SPAD":
        return {"display": "block"}, {"display": "none"}
    return {"display": "none"}, {"display": "block"}




if __name__ == "__main__":
    import warnings
    warnings.filterwarnings("ignore", category=UserWarning)
    app.run(debug=True, port=999)


print(float(samples[0].split("_")[3].split("-")[0]) )